# Flight Maneuver Classification - Test Submission

Generate predictions on test set using the best trained model.

## 1. Setup & Load Model

In [ ]:
import pandas as pd
import numpy as np
import sys
sys.path.insert(0, '..')
from utils import extract_features
import pickle
import json
import warnings
warnings.filterwarnings('ignore')

print("Loading best model...")
with open('../output/best_model.pkl', 'rb') as f:
    model_info = pickle.load(f)

best_model = model_info['model']
model_name = model_info['model_name']
feature_columns = model_info['feature_columns']
validation_results = model_info['validation_results']

print(f"✓ Model loaded: {model_name}")
print(f"  Validation Accuracy: {validation_results['accuracy']:.4f}")
print(f"  Validation Min F1: {validation_results['min_f1']:.4f}")

## 2. Load Test Data

In [2]:
test_df = pd.read_csv('../data/test_set.csv')
print(f"Test set shape: {test_df.shape}")
print(f"Unique maneuvers: {test_df['maneuver_Id'].nunique()}")

Test set shape: (200000, 6)
Unique maneuvers: 6351


## 3. Feature Extraction Function

## 4. Extract Test Features

In [4]:
print("Extracting test features...")
X_test = test_df.groupby('maneuver_Id').apply(extract_features).reset_index()
X_test = X_test.fillna(0)
X_test.columns = X_test.columns.astype(str)
X_test_features = X_test.iloc[:, 1:]

print(f"✓ Test features extracted: {X_test_features.shape}")
print(f"Feature types: {X_test_features.dtypes.unique()}")

Extracting test features...
✓ Test features extracted: (6351, 46)
Feature types: [dtype('float64')]


## 5. Generate Predictions

In [ ]:
print(f"\nGenerating predictions with {model_name}...")
y_pred = best_model.predict(X_test_features)

print(f"✓ Predictions generated")
print(f"\nPrediction distribution:")
print(pd.Series(y_pred).value_counts().sort_index())

## 6. Create & Save Submission

In [ ]:
# Create submission dataframe
submission = pd.DataFrame({
    'maneuver_id': X_test['maneuver_Id'],
    'class': y_pred
})

print(f"Submission shape: {submission.shape}")
print(f"\nFirst 10 rows:")
print(submission.head(10))

# Save submission
submission.to_csv('../output/submission.csv', index=False)
print(f"\n✓ Submission saved to output/submission.csv")

## 7. Summary

In [ ]:
print(f"\n{'='*60}")
print("SUBMISSION COMPLETE")
print(f"{'='*60}")
print(f"Model: {model_name}")
print(f"Test maneuvers: {len(submission)}")
print(f"\nClass distribution in submission:")
for cls in sorted(submission['class'].unique()):
    count = (submission['class'] == cls).sum()
    pct = count / len(submission) * 100
    print(f"  Class {cls}: {count:5d} ({pct:5.1f}%)")
print(f"\nValidation performance:")
print(f"  Accuracy: {validation_results['accuracy']:.4f}")
print(f"  Min F1: {validation_results['min_f1']:.4f}")
print(f"  Per-class F1: {validation_results['f1_scores']}")